# Stain reference patch (Macenko / Reinhard / Vahadane benchmark target)

Loads `experiments/stain_reference/stain_reference.json`, reads `reference_train_index` from the PCam **training** H5, and shows the patch used to fit classical normalizers in `prepare_stain_benchmark_h5.py`.

Run from repo root or from `notebooks/` (kernel cwd must allow finding `experiments/stain_reference/stain_reference.json`).

In [ ]:
from pathlib import Path
import json

import h5py
import matplotlib.pyplot as plt
import numpy as np

# Resolve repo root (works if cwd is GP_ECG or notebooks/)
ROOT = None
for base in (Path("."), Path("..")):
    cfg = base / "experiments" / "stain_reference" / "stain_reference.json"
    if cfg.exists():
        ROOT = base.resolve()
        break
if ROOT is None:
    raise FileNotFoundError(
        "Could not find experiments/stain_reference/stain_reference.json — "
        "set kernel working directory to the repo root or notebooks/."
    )

ref_path = ROOT / "experiments" / "stain_reference" / "stain_reference.json"
with open(ref_path, encoding="utf-8") as f:
    ref_cfg = json.load(f)

ref_idx = int(ref_cfg["reference_train_index"])
train_x = ROOT / "pcam_data" / "training" / "camelyonpatch_level_2_split_train_x.h5"
if not train_x.exists():
    raise FileNotFoundError(f"Missing PCam train H5: {train_x}")

with h5py.File(train_x, "r") as f:
    patch = np.asarray(f["x"][ref_idx])

# PCam H5 is usually uint8 RGB; normalize to float [0,1] for imshow
if patch.dtype == np.uint8 or patch.max() > 1.0:
    display_img = patch.astype(np.float64) / 255.0
else:
    display_img = np.clip(patch.astype(np.float64), 0.0, 1.0)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
ax.imshow(display_img)
ax.set_title(f"reference_train_index = {ref_idx}\n{train_x.name}")
ax.axis("off")
plt.tight_layout()
plt.show()

print("stain_reference.json:", ref_path)
print(json.dumps(ref_cfg, indent=2))